# Inspect `columns_to_drop.pickle`

This notebook loads `SWAN/databases/columns_to_drop.pickle` and prints a readable summary of its contents (type, size, and samples).

In [ ]:
from pathlib import Path
import pickle

path = Path("./databases/columns_to_drop.pickle")
with path.open("rb") as f:
    obj = pickle.load(f)

print("Loaded:", path)
print("Type:", type(obj))


In [ ]:
import pprint

try:
    from IPython.display import display
except Exception:
    display = None


def summarize(o):
    if isinstance(o, dict):
        print(f"dict with {len(o)} keys")
        keys = list(o.keys())
        print("First keys:")
        pprint.pprint(keys)
        for k in keys:
            print(f"- {k!r}:")
            pprint.pprint(o[k])
        return
    else:
        raise ValueError(f"Unsupported type: {type(o)}")

    if isinstance(o, (list, tuple, set)):
        seq = list(o)
        print(f"{type(o).__name__} with {len(seq)} items")
        print("First items:")
        pprint.pprint(seq)
        if seq and all(isinstance(x, str) for x in seq):
            print("\nUnique items:", len(set(seq)))
        return

    if hasattr(o, "shape") and hasattr(o, "head"):
        print("table-like object")
        print("Shape:", getattr(o, "shape", None))
        try:
            print("Columns:")
            pprint.pprint(list(getattr(o, "columns", [])))
        except Exception:
            pass
        head = o.head(20)
        if display is not None:
            display(head)
        else:
            print(head)
        return

    pprint.pprint(o)


summarize(obj)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
from collections import defaultdict, deque

SCHEMA_ROOT = Path("./databases/duckdb/schema")


def load_db_schema(db_id: str, *, schema_root: Path = SCHEMA_ROOT) -> dict[str, dict]:
    """Load per-table JSON schema artifacts for a SWAN duckdb database."""
    db_dir = schema_root / db_id / db_id
    if not db_dir.exists():
        raise FileNotFoundError(f"Schema not found for {db_id!r} at: {db_dir}")

    tables: dict[str, dict] = {}
    for p in sorted(db_dir.glob("*.json")):
        obj = json.loads(p.read_text(encoding="utf-8"))
        # obj has: table_name, table_fullname, column_names, column_types, description, sample_rows
        table = p.stem
        tables[table] = obj

    if not tables:
        raise ValueError(f"No table schemas found under: {db_dir}")

    return tables


def _strip_backticks(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == "`" and s[-1] == "`":
        return s[1:-1]
    return s


_non_alnum = re.compile(r"[^a-z0-9]+")


def norm_name(s: str) -> str:
    """Normalize identifiers for matching across tables."""
    s = _strip_backticks(s).lower()
    return _non_alnum.sub("", s)


def token_set(s: str) -> set[str]:
    s = _strip_backticks(s).lower()
    toks = [t for t in re.split(r"[^a-z0-9]+", s) if t]
    return set(toks)


# Tokens that often add little semantic meaning for attribute equality.
# We DO NOT remove these for key-like columns (id/code/etc).
STOP_TOKENS = {
    "name",
    "type",
    "status",
    "desc",
    "description",
    "number",
    "num",
    "no",
}


def _is_key_like_name(col_name: str) -> bool:
    n = norm_name(col_name)
    return (
        n.endswith("id")
        or n.endswith("code")
        or n in {"id", "pk", "key"}
        or n.endswith("key")
    )


def attr_token_set(col_name: str) -> set[str]:
    toks = token_set(col_name)
    if _is_key_like_name(col_name):
        return toks
    return {t for t in toks if t not in STOP_TOKENS}


def name_similarity(a: str, b: str) -> float:
    """Similarity of normalized names in [0,1]."""
    return SequenceMatcher(None, norm_name(a), norm_name(b)).ratio()


# Parse entries like: schools.`City` or races.date
# Some pickle strings contain accidental concatenations; we extract ALL occurrences.
_entry_pat = re.compile(r"(?P<table>[A-Za-z0-9_]+)\.(?P<col>`[^`]+`|[A-Za-z0-9_]+)")


def extract_table_cols(raw: str) -> list[tuple[str, str]]:
    return [(m.group("table"), m.group("col")) for m in _entry_pat.finditer(raw)]


In [ ]:
@dataclass(frozen=True)
class ColRef:
    table: str
    col: str
    col_type: str


@dataclass
class Candidate:
    other: ColRef
    exact_norm_match: bool
    sim: float
    token_jaccard: float
    sample_overlap: float | None
    join_path: list[tuple[str, str]] | None  # [(table, via_key_norm), ...] includes src/dst steps


def build_column_index(tables: dict[str, dict]) -> tuple[list[ColRef], dict[str, list[ColRef]]]:
    all_cols: list[ColRef] = []
    by_norm: dict[str, list[ColRef]] = defaultdict(list)

    for table, obj in tables.items():
        names = obj["column_names"]
        types = obj["column_types"]
        for cn, ct in zip(names, types):
            ref = ColRef(table=table, col=cn, col_type=ct)
            all_cols.append(ref)
            by_norm[norm_name(cn)].append(ref)

    return all_cols, by_norm


def _resolve_col_key(tables: dict[str, dict], table: str, col: str) -> str | None:
    obj = tables.get(table)
    if not obj:
        return None
    target = norm_name(col)
    for cn in obj.get("column_names", []):
        if norm_name(cn) == target:
            return cn
    return None


def sample_values_text(tables: dict[str, dict], table: str, col: str) -> set[str]:
    obj = tables.get(table)
    if not obj:
        return set()
    key = _resolve_col_key(tables, table, col)
    if not key:
        return set()
    vals = set()
    for row in obj.get("sample_rows", []) or []:
        v = row.get(key)
        if v is None:
            continue
        if isinstance(v, str):
            vv = v.strip()
            if vv:
                vals.add(vv)
    return vals


def sample_overlap_ratio(tables: dict[str, dict], a: ColRef, b: ColRef) -> float | None:
    # Only meaningful for text-like attributes; numeric IDs overlap is usually meaningless.
    if a.col_type != "TEXT" or b.col_type != "TEXT":
        return None
    av = sample_values_text(tables, a.table, a.col)
    bv = sample_values_text(tables, b.table, b.col)
    if not av or not bv:
        return None
    return len(av & bv) / min(len(av), len(bv))


def sample_values_any(
    tables: dict[str, dict],
    table: str,
    col: str,
    *,
    max_values: int = 6,
) -> list[str]:
    """Extract up to `max_values` unique sample values for a column.

    Uses the exported per-table `sample_rows` from `duckdb_export_schema.py`.
    """
    obj = tables.get(table)
    if not obj:
        return []
    key = _resolve_col_key(tables, table, col)
    if not key:
        return []

    seen: set[str] = set()
    out: list[str] = []
    for row in obj.get("sample_rows", []) or []:
        v = row.get(key)
        if v is None:
            continue
        if isinstance(v, str):
            vv = v.strip()
            if not vv:
                continue
        elif isinstance(v, (bool, int, float)):
            vv = str(v)
        else:
            vv = str(v)

        if vv in seen:
            continue
        seen.add(vv)
        out.append(vv)
        if len(out) >= max_values:
            break

    return out


def is_key_like(col_name: str) -> bool:
    n = norm_name(col_name)
    return (
        n.endswith("id")
        or n.endswith("code")
        or n in {"id", "pk", "key"}
        or n.endswith("key")
    )


def infer_join_graph(tables: dict[str, dict]) -> dict[str, list[tuple[str, str]]]:
    """Infer possible joins from shared key-like column names.

    Returns adjacency list: table -> [(other_table, shared_key_norm), ...].
    """
    table_keys: dict[str, set[str]] = {}
    for table, obj in tables.items():
        keys = set()
        for cn, ct in zip(obj["column_names"], obj["column_types"]):
            if not is_key_like(cn):
                continue
            # Favor numeric keys; allow text codes too.
            if ct in {"NUMBER", "TEXT"}:
                keys.add(norm_name(cn))
        table_keys[table] = keys

    adj: dict[str, list[tuple[str, str]]] = {t: [] for t in tables}
    tnames = list(tables.keys())
    for i, t1 in enumerate(tnames):
        for t2 in tnames[i + 1 :]:
            shared = table_keys[t1].intersection(table_keys[t2])
            for k in sorted(shared):
                adj[t1].append((t2, k))
                adj[t2].append((t1, k))

    return adj


def find_join_path(
    adj: dict[str, list[tuple[str, str]]],
    src: str,
    dst: str,
    *,
    max_hops: int = 3,
) -> list[tuple[str, str]] | None:
    """Return a short join path as [(table, via_key_norm), ...]."""
    if src == dst:
        return [(src, "")]

    q = deque([(src, [(src, "")])])
    seen = {src}
    while q:
        cur, path = q.popleft()
        hops = len(path) - 1
        if hops >= max_hops:
            continue
        for nxt, k in adj.get(cur, []):
            if nxt in seen:
                continue
            npath = path + [(nxt, k)]
            if nxt == dst:
                return npath
            seen.add(nxt)
            q.append((nxt, npath))

    return None


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def find_candidates_for_drop(
    *,
    drop: ColRef,
    all_cols: list[ColRef],
    adj: dict[str, list[tuple[str, str]]],
    tables: dict[str, dict],
    top_k: int = 8,
    sim_threshold: float = 0.86,
    token_threshold: float = 0.6,
) -> list[Candidate]:
    dn = norm_name(drop.col)
    dtoks = attr_token_set(drop.col)

    # Columns like `name`/`url` are often entity-specific and not interchangeable across tables.
    generic_text_norms = {"name", "url", "website"}

    out: list[Candidate] = []
    for ref in all_cols:
        if ref.table == drop.table:
            continue
        if ref.col_type != drop.col_type:
            continue

        rn = norm_name(ref.col)
        exact = (rn == dn) and (dn != "")
        sim = SequenceMatcher(None, dn, rn).ratio() if (dn and rn) else 0.0
        rtoks = attr_token_set(ref.col)
        tj = jaccard(dtoks, rtoks)
        containment = (
            (len(dtoks & rtoks) / min(len(dtoks), len(rtoks)))
            if (dtoks and rtoks)
            else 0.0
        )

        # Also catch simple cases like countyname vs county.
        substring = (dn and rn) and (dn in rn or rn in dn)

        if not (
            exact
            or sim >= sim_threshold
            or tj >= token_threshold
            or containment >= 0.9
            or (substring and max(len(dn), len(rn)) >= 6)
        ):
            continue

        ov = sample_overlap_ratio(tables, drop, ref)
        if exact and drop.col_type == "TEXT" and dn in generic_text_norms and ov == 0.0:
            # Same column name but clearly different sample values (e.g., different entity URLs).
            continue

        path = find_join_path(adj, drop.table, ref.table, max_hops=3)

        out.append(
            Candidate(
                other=ref,
                exact_norm_match=exact,
                sim=sim,
                token_jaccard=tj,
                sample_overlap=ov,
                join_path=path,
            )
        )

    out.sort(
        key=lambda c: (
            c.exact_norm_match,
            (c.sample_overlap is not None and c.sample_overlap > 0),
            c.sample_overlap if c.sample_overlap is not None else -1.0,
            c.sim,
            c.token_jaccard,
            c.join_path is not None,
        ),
        reverse=True,
    )
    return out[:top_k]


In [ ]:
def lookup_col_type(tables: dict[str, dict], table: str, col: str) -> str | None:
    obj = tables.get(table)
    if not obj:
        return None
    for cn, ct in zip(obj["column_names"], obj["column_types"]):
        if norm_name(cn) == norm_name(col):
            return ct
    return None


def analyze_db_columns_to_drop(db_id: str, raw_entries: list[str], *, top_k: int = 8):
    tables = load_db_schema(db_id)
    all_cols, _by_norm = build_column_index(tables)
    adj = infer_join_graph(tables)

    drops: list[ColRef] = []
    unparsable: list[str] = []
    missing_tables: set[str] = set()

    for raw in raw_entries:
        pairs = extract_table_cols(raw)
        if not pairs:
            unparsable.append(raw)
            continue
        for table, col in pairs:
            if table not in tables:
                missing_tables.add(table)
            ct = lookup_col_type(tables, table, col)
            if ct is None:
                # Keep it, but mark unknown type; we will still try name-based matches.
                ct = "?"
            drops.append(ColRef(table=table, col=_strip_backticks(col), col_type=ct))

    # If we had unknown types, allow matching across any type for those.
    results: list[dict] = []
    for d in drops:
        if d.col_type == "?":
            # widen: run candidate search across all cols
            d_all_cols = [c for c in all_cols if c.table != d.table]
            # fake type filtering by setting drop col_type to each candidate's type inside loop
            cands = []
            for ref in d_all_cols:
                dd = ColRef(table=d.table, col=d.col, col_type=ref.col_type)
                cands.extend(
                    find_candidates_for_drop(
                        drop=dd,
                        all_cols=all_cols,
                        adj=adj,
                        tables=tables,
                        top_k=top_k,
                    )
                )
            # keep top unique other cols
            seen = set()
            uniq = []
            for c in sorted(cands, key=lambda c: (c.exact_norm_match, c.sim, c.token_jaccard, c.join_path is not None), reverse=True):
                k = (c.other.table, norm_name(c.other.col))
                if k in seen:
                    continue
                seen.add(k)
                uniq.append(c)
                if len(uniq) >= top_k:
                    break
            cands = uniq
        else:
            cands = find_candidates_for_drop(
                drop=d,
                all_cols=all_cols,
                adj=adj,
                tables=tables,
                top_k=top_k,
            )

        drop_samples = sample_values_any(tables, d.table, d.col)

        if not cands:
            results.append(
                {
                    "db": db_id,
                    "drop_table": d.table,
                    "drop_col": d.col,
                    "drop_type": d.col_type,
                    "drop_samples": drop_samples,
                    "candidate_count": 0,
                    "candidates": [],
                }
            )
            continue

        results.append(
            {
                "db": db_id,
                "drop_table": d.table,
                "drop_col": d.col,
                "drop_type": d.col_type,
                "drop_samples": drop_samples,
                "candidate_count": len(cands),
                "candidates": [
                    {
                        "other_table": c.other.table,
                        "other_col": c.other.col,
                        "other_type": c.other.col_type,
                        "other_samples": sample_values_any(tables, c.other.table, c.other.col),
                        "exact_norm_match": c.exact_norm_match,
                        "name_sim": round(c.sim, 3),
                        "token_jaccard": round(c.token_jaccard, 3),
                        "sample_overlap": None if c.sample_overlap is None else round(c.sample_overlap, 3),
                        "join_possible": c.join_path is not None,
                        "join_path": c.join_path,
                    }
                    for c in cands
                ],
            }
        )

    return {
        "db": db_id,
        "drop_count": len(drops),
        "unparsable": unparsable,
        "missing_tables": sorted(missing_tables),
        "results": results,
    }


analysis = {db: analyze_db_columns_to_drop(db, entries) for db, entries in obj.items()}

# Pretty-print a compact summary
for db_id, rep in analysis.items():
    print("\n===", db_id, "===")
    if rep["unparsable"]:
        print("Unparsable entries:")
        for x in rep["unparsable"]:
            print(" -", x)

    if rep.get("missing_tables"):
        print("Missing tables (not found in exported schema):")
        for t in rep["missing_tables"]:
            print(" -", t)

    for r in rep["results"]:
        if r["candidate_count"] == 0:
            continue
        print(f"\nDROP {r['drop_table']}.{r['drop_col']} ({r['drop_type']})")
        print("  drop_samples:", r.get("drop_samples"))
        for c in r["candidates"]:
            jp = c["join_path"]
            jp_str = None
            if jp:
                # show as: A -k-> B -k-> C
                parts = [jp[0][0]]
                for t, k in jp[1:]:
                    parts.append(f"-({k})-> {t}")
                jp_str = " ".join(parts)
            print(
                "  ->",
                f"{c['other_table']}.{c['other_col']} ({c['other_type']})",
                f"exact={c['exact_norm_match']}",
                f"sim={c['name_sim']}",
                f"tok={c['token_jaccard']}",
                f"ov={c.get('sample_overlap')}",
                f"join={c['join_possible']}",
                ("path=" + jp_str) if jp_str else "",
            )
            print("     other_samples:", c.get("other_samples"))


In [ ]:
# Build a flat table for easier filtering/sorting
flat_rows = []
for db_id, rep in analysis.items():
    for r in rep["results"]:
        if not r["candidates"]:
            flat_rows.append(
                {
                    "db": db_id,
                    "drop_table": r["drop_table"],
                    "drop_col": r["drop_col"],
                    "drop_type": r["drop_type"],
                    "other_table": None,
                    "other_col": None,
                    "other_type": None,
                    "exact_norm_match": False,
                    "name_sim": None,
                    "token_jaccard": None,
                    "sample_overlap": None,
                    "drop_samples": r.get("drop_samples"),
                    "other_samples": None,
                    "join_possible": False,
                    "join_path": None,
                }
            )
            continue
        for c in r["candidates"]:
            flat_rows.append(
                {
                    "db": db_id,
                    "drop_table": r["drop_table"],
                    "drop_col": r["drop_col"],
                    "drop_type": r["drop_type"],
                    "other_table": c["other_table"],
                    "other_col": c["other_col"],
                    "other_type": c["other_type"],
                    "exact_norm_match": c["exact_norm_match"],
                    "name_sim": c["name_sim"],
                    "token_jaccard": c["token_jaccard"],
                    "sample_overlap": c.get("sample_overlap"),
                    "drop_samples": r.get("drop_samples"),
                    "other_samples": c.get("other_samples"),
                    "join_possible": c["join_possible"],
                    "join_path": c["join_path"],
                }
            )

try:
    import pandas as pd

    df = pd.DataFrame(flat_rows)

    # Drop-columns that appear elsewhere AND are join-recoverable (heuristic)
    recoverable = (
        df[df["other_table"].notna()]
        .groupby(["db", "drop_table", "drop_col", "drop_type"], as_index=False)
        .agg(
            any_exact=("exact_norm_match", "max"),
            any_joinable=("join_possible", "max"),
            max_sample_overlap=("sample_overlap", "max"),
            max_sim=("name_sim", "max"),
            max_tok=("token_jaccard", "max"),
            n_candidates=("other_table", "count"),
        )
        .sort_values(
            ["db", "any_joinable", "any_exact", "max_sample_overlap", "max_sim", "n_candidates"],
            ascending=[True, False, False, False, False, False],
        )
    )

    display(recoverable)
except Exception as e:
    print("pandas not available (or display not available). Showing first 25 flat rows.")
    print("Error:", e)
    for row in flat_rows[:25]:
        print(row)


In [ ]:
import re
import pickle
from pathlib import Path

db_id = "california_schools"
to_add = [
    "schools.FundingType",
    "schools.Charter",
    "schools.County",
]

IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def ensure_backticked(table: str, col: str) -> str:
    col = col.strip()
    if col.startswith("`") and col.endswith("`"):
        return f"{table}.{col}"
    return f"{table}.`{col}`"


def _strip_bt(s: str) -> str:
    s = (s or "").strip()
    return s[1:-1] if len(s) >= 2 and s[0] == "`" and s[-1] == "`" else s


def _norm_entry(entry: str) -> tuple[str, str]:
    table, col = entry.split(".", 1)
    return table.strip().lower(), _strip_bt(col).strip().lower()


def _fmt_entry(table: str, col: str) -> str:
    col = _strip_bt(col).strip()
    col_part = col if IDENT.match(col) else f"`{col}`"
    return f"{table.strip()}.{col_part}"


obj.setdefault(db_id, [])
existing = {_norm_entry(x) for x in obj[db_id] if isinstance(x, str)}

added = []
for entry in to_add:
    t, c = entry.split(".", 1)
    k = (t.strip().lower(), _strip_bt(c).strip().lower())
    print(f"Adding {t}.{c} -> {ensure_backticked(t, c)}")
    s = ensure_backticked(t, c)
    obj[db_id].append(s)
    existing.add(k)
    added.append(s)

new_path = Path(path).with_name("columns_to_drop_extra.pickle")
with new_path.open("wb") as f:
    pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Added:", added)
print("Wrote:", new_path)

# (optional) verify it loads
with new_path.open("rb") as f:
    obj2 = pickle.load(f)
obj2